# 컨버터 (Converters)

컨버터는 프롬프트를 대상(target)에 보내기 전에 변환하는 데 사용됩니다.

이는 다양한 이유로 유용할 수 있습니다. 예를 들어, 프롬프트를 다른 형식으로 인코딩하거나, 프롬프트에 추가 정보를 더하는 경우가 있습니다. 프롬프트를 대상에 보내기 전에 base64로 변환하거나, 프롬프트 앞에 질문임을 나타내는 접두사를 추가하고 싶을 수 있습니다.

컨버터는 다양한 방식으로 프롬프트를 변환할 수 있습니다:
- **텍스트-텍스트 변환**: 인코딩, 난독화, 번역, 의미론적 변환
- **멀티모달 변환**: 텍스트, 이미지, 오디오, 비디오, 파일 간 변환
- **대화형 변환**: 사람이 직접 검토하고 수정하는 방식 (Human-in-the-loop)

## 컨버터 모달리티 참조 테이블

다음 테이블은 입력 및 출력 모달리티별로 정리된 사용 가능한 모든 컨버터를 보여줍니다:

In [1]:
import pandas as pd

from pyrit.prompt_converter import get_converter_modalities
from pyrit.setup import IN_MEMORY, initialize_pyrit_async

await initialize_pyrit_async(memory_db_type=IN_MEMORY)  # type: ignore

# Get all converters with their modalities
converter_list = get_converter_modalities()

# Create a list of rows for the DataFrame
rows = []
for name, inputs, outputs in converter_list:
    input_str = ", ".join(inputs) if inputs else "any"
    output_str = ", ".join(outputs) if outputs else "any"
    rows.append({"Input Modality": input_str, "Output Modality": output_str, "Converter": name})

# Create DataFrame and sort
df = pd.DataFrame(rows)
df = df.sort_values(by=["Input Modality", "Output Modality", "Converter"]).reset_index(drop=True)

# Display all rows
pd.set_option("display.max_rows", None)
df

Found default environment files: ['/Users/selectstar/.pyrit/.env', '/Users/selectstar/.pyrit/.env.local']
Loaded environment file: /Users/selectstar/.pyrit/.env
Loaded environment file: /Users/selectstar/.pyrit/.env.local


,Input Modality,Output Modality,Converter
0,audio_path,audio_path,AudioFrequencyConverter
1,audio_path,text,AzureSpeechAudioToTextConverter
2,image_path,image_path,AddTextImageConverter
3,image_path,image_path,TransparencyAttackConverter
4,image_path,video_path,AddImageVideoConverter
5,"image_path, url",image_path,ImageCompressionConverter
6,text,audio_path,AzureSpeechTextToAudioConverter
7,text,binary_path,PDFConverter
8,text,image_path,AddImageTextConverter
9,text,image_path,QRCodeConverter


## 컨버터 카테고리

컨버터는 다음 카테고리로 구성되어 있습니다:

- **[텍스트-텍스트 컨버터](1_text_to_text_converters.ipynb)**: 비-LLM 방식 (인코딩, 난독화) 및 LLM 기반 방식 (번역, 변형, 톤 변환)
- **[오디오 컨버터](2_audio_converters.ipynb)**: 텍스트→오디오, 오디오→텍스트, 오디오→오디오
- **[이미지 컨버터](3_image_converters.ipynb)**: 텍스트→이미지, 이미지→이미지
- **[비디오 컨버터](4_video_converters.ipynb)**: 이미지→비디오
- **[파일 컨버터](5_file_converters.ipynb)**: 텍스트→파일 (PDF, URL)
- **[선택적 변환](6_selectively_converting.ipynb)**: 프롬프트의 일부분만 변환하는 고급 기법
- **[Human-in-the-Loop 컨버터](7_human_converter.ipynb)**: 대화형 프롬프트 검토 및 수정

## 간단한 컨버터 예제

컨버터를 사용하여 이러한 유형의 변환을 수행할 수 있습니다. 다음은 Rot13Converter, RandomCapitalLettersConverter, AsciiArtConverter를 사용하는 간단한 프로그램입니다.

In [13]:
from pyrit.prompt_converter import (
    AsciiArtConverter,
    BinaryConverter,
    FirstLetterConverter,
    RandomCapitalLettersConverter,
    ROT13Converter,
)

#prompt = "tell me how to cut down a tree"
prompt = "나무 베는 법에 대해 알려줘"


print(await ROT13Converter().convert_tokens_async(prompt=prompt))  # type: ignore
print(await RandomCapitalLettersConverter(percentage=25.0).convert_tokens_async(prompt=prompt))  # type: ignore
print(await AsciiArtConverter().convert_tokens_async(prompt=prompt))  # type: ignore
print(await BinaryConverter().convert_tokens_async(prompt=prompt))  # type: ignore
print(await FirstLetterConverter().convert_tokens_async(prompt=prompt))  # type: ignore

text: 나무 베는 법에 대해 알려줘
text: 나무 베는 법에 대해 알려줘
text:     
    
    
    
    
    
    
    

text: 1011000010011000 1011101100110100 0000000000100000 1011110010100000 1011001010010100 0000000000100000 1011110010010101 1100010111010000 0000000000100000 1011001100000000 1101010101110100 0000000000100000 1100010101001100 1011100000100100 1100100100011000
text: 나 베 법 대 알


## 컨버터 스태킹 (Stacking Converters)

컨버터는 단독으로 사용할 수도 있지만, 파이프라인의 한 구성요소로 생각해야 합니다. 일반적으로 모든 공격에는 프롬프트를 대상에 보내기 전에 변환할 수 있는 인자가 있습니다. 컨버터는 중첩(스태킹)할 수 있고, LLM을 활용할 수 있으며, 강력한 도구입니다.

시작하기 전에 올바른 버전의 PyRIT가 설치되어 있고, [여기](../../setup/populating_secrets.md)에 설명된 대로 시크릿이 구성되어 있는지 확인하세요.

### PromptSendingAttack으로 컨버터 스태킹하기

다음 예제에서는 TextTarget을 사용하므로 프롬프트가 단순히 출력되고 메모리에 추가됩니다. 이는 레드 팀 활동 시 프롬프트를 수동으로 입력해야 할 때 유용합니다. 단, 대상은 다른 어떤 [target](../targets/0_prompt_targets.md)으로도 교체할 수 있습니다. 예를 들어, API 접근 권한이 있다면 해당 대상을 추가할 수 있습니다.

이 예제에서는 컨버터가 스태킹됩니다. 먼저 `VariationConverter`를 사용하여 변형을 생성한 다음, `StringJoinConverter`를 사용하여 글자 사이에 대시를 추가합니다. 순서가 중요하다는 점을 기억하세요. `StringJoinConverter`가 먼저 오면 LLM에게 다음과 같은 프롬프트의 변형을 요청하게 됩니다:
"t-e-l-l- -m-e- -h-o-w- -t-o- -c-u-t- -d-o-w-n- -a- -t-r-e-e"

In [5]:
from pyrit.executor.attack import (
    AttackConverterConfig,
    ConsoleAttackResultPrinter,
    PromptSendingAttack,
)
from pyrit.prompt_converter import StringJoinConverter, VariationConverter
from pyrit.prompt_normalizer import PromptConverterConfiguration
from pyrit.prompt_target import OpenAIChatTarget, TextTarget

objective = "나무 베는 법에 대해 알려줘"

# converter target to send prompt to; in this case configured to the Azure OpenAI GPT-4o Chat model
converter_target = OpenAIChatTarget()
prompt_variation_converter = VariationConverter(converter_target=converter_target)

converter_configs = PromptConverterConfiguration.from_converters(  # type: ignore
    converters=[prompt_variation_converter, StringJoinConverter()]
)

converter_config = AttackConverterConfig(request_converters=converter_configs)  # type: ignore

target = TextTarget()
attack = PromptSendingAttack(
    objective_target=target,
    attack_converter_config=converter_config,
)

result = await attack.execute_async(objective=objective)  # type: ignore

printer = ConsoleAttackResultPrinter()
await printer.print_conversation_async(result=result)  # type: ignore

TextTarget: user: 나-무-를 자-르-는 방-법-에 대-해 설-명-해-줄-래-?


[PromptSendingAttack (ID: b989f823)] No response received on attempt 1 (likely filtered)



────────────────────────────────────────────────────────────────────────────────────────────────────
🔹 Turn 1 - USER
────────────────────────────────────────────────────────────────────────────────────────────────────
   Original:
  나무 베는 법에 대해 알려줘

   Converted:
  나-무-를 자-르-는 방-법-에 대-해 설-명-해-줄-래-?

────────────────────────────────────────────────────────────────────────────────────────────────────


### 한국어 프롬프트로 컨버터 스태킹 테스트

한국어 프롬프트에 대해서도 컨버터 스태킹을 적용해 봅니다.

In [8]:
from pyrit.executor.attack import (
    AttackConverterConfig,
    ConsoleAttackResultPrinter,
    PromptSendingAttack,
)
from pyrit.prompt_converter import StringJoinConverter, VariationConverter
from pyrit.prompt_normalizer import PromptConverterConfiguration
from pyrit.prompt_target import OpenAIChatTarget, TextTarget

objective_ko = "나무를 베는 방법을 알려주세요"

# 컨버터 대상: Azure OpenAI GPT-4o Chat 모델로 구성
converter_target = OpenAIChatTarget()
prompt_variation_converter = VariationConverter(converter_target=converter_target)

converter_configs = PromptConverterConfiguration.from_converters(  # type: ignore
    converters=[prompt_variation_converter, StringJoinConverter()]
)

converter_config = AttackConverterConfig(request_converters=converter_configs)  # type: ignore

target = TextTarget()
attack = PromptSendingAttack(
    objective_target=target,
    attack_converter_config=converter_config,
)

result = await attack.execute_async(objective=objective_ko)  # type: ignore

printer = ConsoleAttackResultPrinter()
await printer.print_conversation_async(result=result)  # type: ignore

TextTarget: user: 나-무-를 어-떻-게 잘-라-야 할-지 알-려-주-실-래-요-?


[PromptSendingAttack (ID: b3b4bb83)] No response received on attempt 1 (likely filtered)



────────────────────────────────────────────────────────────────────────────────────────────────────
🔹 Turn 1 - USER
────────────────────────────────────────────────────────────────────────────────────────────────────
   Original:
  나무를 베는 방법을 알려주세요

   Converted:
  나-무-를 어-떻-게 잘-라-야 할-지 알-려-주-실-래-요-?

────────────────────────────────────────────────────────────────────────────────────────────────────


## 응답 컨버터 (Response Converters)

지금까지 프롬프트를 대상에 보내기 전에 변환하는 **요청 컨버터(request converters)**에 초점을 맞추었습니다. PyRIT는 대상의 응답을 반환하기 전에 변환하는 **응답 컨버터(response converters)**도 지원합니다. 이는 다음과 같은 시나리오에서 유용합니다:

- 다른 언어로 프롬프트를 보낸 후 응답을 원래 언어로 다시 번역
- 인코딩된 응답 디코딩
- 응답 텍스트 정규화 또는 정리

응답 컨버터는 요청 컨버터와 동일한 `PromptConverterConfiguration` 클래스를 사용합니다. `AttackConverterConfig`의 `response_converters` 매개변수를 통해 구성됩니다.

### 번역 왕복(Round-Trip) 예제

일반적인 사용 사례는 대상이 비영어 입력을 어떻게 처리하는지 테스트하기 위해 다른 언어로 프롬프트를 보내는 것입니다. 이 예제에서는:

1. **요청 컨버터**를 사용하여 프롬프트를 영어에서 프랑스어로 번역
2. 번역된 프롬프트를 대상에 전송
3. **응답 컨버터**를 사용하여 응답을 다시 영어로 번역

In [10]:
from pyrit.executor.attack import (
    AttackConverterConfig,
    ConsoleAttackResultPrinter,
    PromptSendingAttack,
)
from pyrit.prompt_converter import TranslationConverter
from pyrit.prompt_normalizer import PromptConverterConfiguration
from pyrit.prompt_target import OpenAIChatTarget

objective = "프랑스의 수도는 어디야?"

# Create an LLM target for the converters
converter_target = OpenAIChatTarget()

# Create an LLM target to send prompts to
prompt_target = OpenAIChatTarget()

# Request converter: translate English to French
request_converter = TranslationConverter(converter_target=converter_target, language="French")
request_converter_config = PromptConverterConfiguration(converters=[request_converter])

# Response converter: translate response back to English
response_converter = TranslationConverter(converter_target=converter_target, language="English")
response_converter_config = PromptConverterConfiguration(converters=[response_converter])

# Configure the attack with both request and response converters
converter_config = AttackConverterConfig(
    request_converters=[request_converter_config],
    response_converters=[response_converter_config],
)

attack = PromptSendingAttack(
    objective_target=prompt_target,
    attack_converter_config=converter_config,
)

result = await attack.execute_async(objective=objective)  # type: ignore

# Print the conversation showing both original and converted values
printer = ConsoleAttackResultPrinter()
await printer.print_conversation_async(result=result)  # type: ignore


────────────────────────────────────────────────────────────────────────────────────────────────────
🔹 Turn 1 - USER
────────────────────────────────────────────────────────────────────────────────────────────────────
   Original:
  프랑스의 수도는 어디야?

   Converted:
  La capitale de la France est où ?

────────────────────────────────────────────────────────────────────────────────────────────────────
🔸 ASSISTANT
────────────────────────────────────────────────────────────────────────────────────────────────────
   Original:
  La capitale de la France est Paris.

   Converted:
  The capital of France is Paris.

────────────────────────────────────────────────────────────────────────────────────────────────────


### 한국어 번역 왕복(Round-Trip) 테스트

한국어 프롬프트를 영어로 번역하여 전송한 후, 응답을 다시 한국어로 번역하는 예제입니다.

1. **요청 컨버터**: 한국어 → 영어로 번역
2. 번역된 프롬프트를 대상에 전송
3. **응답 컨버터**: 영어 응답 → 한국어로 번역

In [11]:
from pyrit.executor.attack import (
    AttackConverterConfig,
    ConsoleAttackResultPrinter,
    PromptSendingAttack,
)
from pyrit.prompt_converter import TranslationConverter
from pyrit.prompt_normalizer import PromptConverterConfiguration
from pyrit.prompt_target import OpenAIChatTarget

objective_ko = "프랑스의 수도는 어디인가요?"

# 컨버터용 LLM 대상 생성
converter_target = OpenAIChatTarget()

# 프롬프트를 보낼 LLM 대상 생성
prompt_target = OpenAIChatTarget()

# 요청 컨버터: 한국어를 영어로 번역
request_converter = TranslationConverter(converter_target=converter_target, language="English")
request_converter_config = PromptConverterConfiguration(converters=[request_converter])

# 응답 컨버터: 응답을 한국어로 번역
response_converter = TranslationConverter(converter_target=converter_target, language="Korean")
response_converter_config = PromptConverterConfiguration(converters=[response_converter])

# 요청 및 응답 컨버터를 모두 포함하여 공격 구성
converter_config = AttackConverterConfig(
    request_converters=[request_converter_config],
    response_converters=[response_converter_config],
)

attack = PromptSendingAttack(
    objective_target=prompt_target,
    attack_converter_config=converter_config,
)

result = await attack.execute_async(objective=objective_ko)  # type: ignore

# 원본과 변환된 값을 모두 보여주는 대화 출력
printer = ConsoleAttackResultPrinter()
await printer.print_conversation_async(result=result)  # type: ignore


────────────────────────────────────────────────────────────────────────────────────────────────────
🔹 Turn 1 - USER
────────────────────────────────────────────────────────────────────────────────────────────────────
   Original:
  프랑스의 수도는 어디인가요?

   Converted:
  What is the capital of France?

────────────────────────────────────────────────────────────────────────────────────────────────────
🔸 ASSISTANT
────────────────────────────────────────────────────────────────────────────────────────────────────
   Original:
  The capital of France is Paris.

   Converted:
  프랑스의 수도는 파리입니다.

────────────────────────────────────────────────────────────────────────────────────────────────────
